<a href="https://colab.research.google.com/github/anirbansen2709/AI-ML_Assignments/blob/main/aci_Assignment1_PS12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Local Beam Search — Optimal Planner
Group   : G028
Problem : Find optimal route from Start (S) to Goal (G) avoiding blocked (X) cells.
Search  : Local Beam Search with k beam states.
Heuristic: Manhattan Distance  h(n) = |x_g - x_n| + |y_g - y_n|
Cost    : Each move costs 1 unit (Up / Down / Left / Right only).
"""
from __future__ import annotations

import os
import sys

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
PROBLEM_SET_ID = "PS12"
INPUT_FILE = f"input{PROBLEM_SET_ID}.txt"
OUTPUT_FILE = f"output{PROBLEM_SET_ID}.txt"

# Cell symbols used in the grid file
EMPTY    = "."
START_CELL = "S"
GOAL_CELL  = "G"
BLOCKED  = "X"

# Movement vectors: Up, Down, Left, Right (row-delta, col-delta)
DIRECTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]


# ---------------------------------------------------------------------------
# Tee — mirrors every print() to both stdout and an output file
# ---------------------------------------------------------------------------
class Tee:
    """
    Wraps sys.stdout so that all output is written to both the console and
    a file simultaneously.
    """

    def __init__(self, filename: str):
        self._console = sys.stdout
        try:
            self._file = open(filename, "w", encoding="utf-8")
        except OSError as exc:
            raise RuntimeError(f"Cannot open output file '{filename}': {exc}") from exc

    def write(self, data: str) -> None:
        self._console.write(data)
        self._file.write(data)

    def flush(self) -> None:
        self._console.flush()
        self._file.flush()

    def close(self) -> None:
        sys.stdout = self._console
        self._file.close()


# ---------------------------------------------------------------------------
# Beam — bounded data structure that holds at most `capacity` beam states
# ---------------------------------------------------------------------------
class BeamFullError(Exception):
    """Raised when insertion is attempted on a full beam."""
    pass


class BeamEmptyError(Exception):
    """Raised when deletion is attempted on an empty beam."""
    pass


# ---------------------------------------------------------------------------
# Beam — bounded data structure holding at most capacity states
# ---------------------------------------------------------------------------
class Beam:
    """
    Fixed-capacity Beam data structure for Local Beam Search.

    Each entry is:
        (state, path)
        state is a tuple like (2, 3) (current row, col).
        path is a list of all coordinates taken to get there: [(0,0), (0,1), (1,1)...].

    state:
        A (row, column) position.

    path:
        The sequence of positions from Start to the current state.
    """

    def __init__(self, capacity: int):
        if not isinstance(capacity, int):
            raise TypeError("Beam capacity must be an integer.")

        if capacity <= 0:
            raise ValueError(
                f"Beam capacity must be greater than zero, got {capacity}."
            )

        self._capacity = capacity
        self._entries = []

    def insert(self, entry: tuple) -> None:
        """
        Insert one entry into the Beam.

        Raises:
            BeamFullError: When the Beam is already full.
        """
        if self.is_full():
            raise BeamFullError(
                f"[Beam] INSERT FAILED — beam is full "
                f"(capacity={self._capacity}). "
                f"Cannot insert state {entry[0]}."
            )

        self._entries.append(entry)

    def remove(self) -> tuple:
        """
        Remove and return the first Beam entry.

        Raises:
            BeamEmptyError: When the Beam is empty.
        """
        if self.is_empty():
            raise BeamEmptyError(
                "[Beam] REMOVE FAILED — beam is empty. "
                "No state is available for deletion."
            )

        return self._entries.pop(0)

    def clear(self) -> None:
        """Remove all Beam entries."""
        self._entries.clear()

    def is_empty(self) -> bool:
        return len(self._entries) == 0

    def is_full(self) -> bool:
        return len(self._entries) >= self._capacity

    @property
    def capacity(self) -> int:
        return self._capacity

    def __len__(self) -> int:
        return len(self._entries)

    def __iter__(self):
        return iter(self._entries)

    def __repr__(self) -> str:
        states = [entry[0] for entry in self._entries]
        return (
            f"Beam(capacity={self._capacity}, "
            f"states={states})"
        )

# ---------------------------------------------------------------------------
# Heuristic
# ---------------------------------------------------------------------------
def manhattan_distance(state: tuple, goal: tuple) -> int:
    """
    Manhattan Distance heuristic.

    h(n) = |row_goal - row_n| + |col_goal - col_n|

    This is admissible for a grid where each move costs 1 and only
    horizontal / vertical moves are allowed.
    """
    return abs(goal[0] - state[0]) + abs(goal[1] - state[1])


# ---------------------------------------------------------------------------
# Successor generation
# ---------------------------------------------------------------------------
def get_successors(
    state: tuple,
    path: list,
    grid: list[list[str]],
    rows: int,
    cols: int,
) -> list[tuple]:
    """
    Return all valid neighbors of `state` that are:
      1. Within the grid bounds.
      2. Not a blocked cell (BLOCKED = 'X').
      3. Not already visited in the current path (prevents cycles without
         maintaining a global visited set, preserving Local Beam Search semantics).

    Directions explored: Up, Down, Left, Right (no diagonal moves).
    """
    path_set = set(path)          # O(1) membership test
    neighbors = []

    for (dr, dc) in DIRECTIONS:
        nr, nc = state[0] + dr, state[1] + dc
        if nr < 0 or nr >= rows or nc < 0 or nc >= cols:
            continue                              # out of bounds
        if grid[nr][nc] == BLOCKED:
            continue                              # obstacle
        if (nr, nc) in path_set:
            continue                              # already visited in this path
        neighbors.append((nr, nc))

    return neighbors


# ---------------------------------------------------------------------------
# Local Beam Search
# ---------------------------------------------------------------------------
def local_beam_search(
    grid: list[list[str]],
    start: tuple,
    goal: tuple,
    k: int,
) -> tuple[list | None, int]:
    """
    Local Beam Search algorithm.

    Maintains k candidate states (the "beam") at each iteration.
    At every step:
      1. Generate all successors of all k beam states.
      2. Remove duplicate states (keeping the first path found).
      3. Compute heuristic h(n) = Manhattan Distance for each candidate.
      4. Select the best k candidates (lowest heuristic value).
      5. If the goal is among the candidates, return the path immediately.

    Args:
        grid  : 2-D list of cell symbols.
        start : (row, col) start position.
        goal  : (row, col) goal position.
        k     : beam width.

    Returns:
        (path, cost) on success, or (None, -1) on failure.
    """
    rows = len(grid)
    cols = len(grid[0])

    # Safety and Sanity Checks for Bounding and Obstacle
    if not (0 <= start[0] < rows and 0 <= start[1] < cols):
        raise ValueError(f"Start position {start} is outside the grid.")
    if not (0 <= goal[0] < rows and 0 <= goal[1] < cols):
        raise ValueError(f"Goal position {goal} is outside the grid.")
    if grid[start[0]][start[1]] == BLOCKED:
        raise ValueError(f"Start position {start} is a blocked cell.")
    if grid[goal[0]][goal[1]] == BLOCKED:
        raise ValueError(f"Goal position {goal} is a blocked cell.")

    # ---- Initialise beam with the start state ----
    beam = Beam(capacity=k)

    try:
        # inserting current state and path
        beam.insert((start, [start]))
    except BeamFullError as exc:
        print(f"[ERROR] {exc}")
        return None, -1

    iteration = 0
    separator = "-" * 60

    print(separator)
    print(f"Initialisation")
    print(f"  Start : {start}   h = {manhattan_distance(start, goal)}")
    print(f"  Beam  : {[entry[0] for entry in beam]}")
    print(separator)

    while not beam.is_empty():
        iteration += 1

        current_entries = list(beam)
        current_states  = [e[0] for e in current_entries]
        current_h       = [manhattan_distance(s, goal) for s in current_states]

        print(f"\n{'=' * 60}")
        print(f"Iteration {iteration}")
        print(f"  Current beam states : {current_states}")
        print(f"  Heuristic values    : {current_h}")

        # ---- Generate all successors ----
        raw_candidates: list[tuple] = []
        for state, path in current_entries:
            for successor in get_successors(state, path, grid, rows, cols):
                new_path = path + [successor]
                raw_candidates.append((successor, new_path))

        # ---- Remove duplicates (first path to each state wins) ----
        seen: set = set()
        unique_candidates: list[tuple] = []
        for state, path in raw_candidates:
            if state not in seen:
                seen.add(state)
                unique_candidates.append((state, path))

        print(f"\n  Successors generated (unique) : "
              f"{[c[0] for c in unique_candidates]}")

        if not unique_candidates:
            print("\n  [!] No successors available. Search failed — no path exists.")
            return None, -1

        # ---- Compute heuristic and sort (ascending = best first) ----
        unique_candidates.sort(key=lambda entry: manhattan_distance(entry[0], goal))

        candidate_states = [c[0] for c in unique_candidates]
        candidate_h      = [manhattan_distance(c[0], goal) for c in unique_candidates]
        print(f"  After sorting by heuristic    : {candidate_states}")
        print(f"  Heuristic values              : {candidate_h}")

        # ---- Check for goal in candidates ----
        for state, path in unique_candidates:
            if state == goal:
                cost = len(path) - 1
                print(f"\n{'=' * 60}")
                print(f"  GOAL REACHED at iteration {iteration}!")
                print(f"{'=' * 60}")
                print(f"\n  Final path (step-by-step):")
                for step_idx, pos in enumerate(path):
                    label = ""
                    if pos == start:
                        label = "  <- START"
                    elif pos == goal:
                        label = "  <- GOAL"
                    print(f"    Step {step_idx:>2}: {pos}{label}")
                arrow_path = " -> ".join(str(p) for p in path)
                print(f"\n  Path  : {arrow_path}")
                print(f"  Cost  : {cost} move(s)")
                return path, cost

        # ---- Select best k states ----
        best_k = unique_candidates[:k]
        beam.clear()

        for entry in best_k:
            try:
                beam.insert(entry)
            except BeamFullError as exc:
                print(f"[ERROR] {exc}")
                return None, -1
            selected_states = [entry[0] for entry in best_k]
        selected_h = [
            manhattan_distance(entry[0], goal)
            for entry in best_k
        ]

        print(
            f"\n  Selected k={k} beam state(s) : "
            f"{selected_states}"
        )
        print(
            f"  Heuristic values             : "
            f"{selected_h}"
        )

    print("\n[!] Beam exhausted without finding the goal.")
    return None, -1


# ---------------------------------------------------------------------------
# Input reader
# ---------------------------------------------------------------------------
def read_input(filename: str) -> tuple[list, tuple, tuple, int]:
    """
    Parse the input file and return (grid, start, goal, k).

    Expected file format:
        Line 1        : <rows> <cols>
        Lines 2..R+1  : space-separated row of cell symbols (S / G / X / .)
        Last line     : k  (beam width, positive integer)

    Raises FileNotFoundError if the file does not exist.
    Raises ValueError on malformed content.
    """
    if not os.path.exists(filename):
        raise FileNotFoundError(
            f"Input file '{filename}' not found. "
            "Please ensure the file is in the same directory as this script."
        )

    # 1. Read all non-empty lines into a list
    with open(filename, "r", encoding="utf-8") as fh:
        raw_lines = [line.strip() for line in fh if line.strip()]

    # 2. Check for minimum required lines
    if len(raw_lines) < 3:
        raise ValueError("Input file has too few lines. Expected: dimensions, grid rows, k.")

    # 3. Parse grid dimensions from the first line
    dim_parts = raw_lines[0].split()
    if len(dim_parts) != 2:
        raise ValueError(f"Line 1 must contain exactly 'rows cols', got: '{raw_lines[0]}'")
    rows, cols = int(dim_parts[0]), int(dim_parts[1])

    # 4. Check if we have enough lines for the grid + k value
    if len(raw_lines) < rows + 2:
        raise ValueError(
            f"Expected {rows} grid row(s) plus a k-value line, "
            f"but only {len(raw_lines) - 1} non-empty lines follow the dimension line."
        )

    # Parse grid
    grid:  list[list[str]] = []
    start: tuple | None = None
    goal:  tuple | None = None

    # 5. Build the grid row by row
    for r in range(rows):
        cells = raw_lines[1 + r].split()
        if len(cells) != cols:
            raise ValueError(
                f"Row {r} has {len(cells)} cell(s), expected {cols}."
            )
        grid.append(cells) # Add the row to our 2D array

        # 6. Scan the row for Start (S) and Goal (G)
        for c, cell in enumerate(cells):
            if cell == START_CELL:
                if start is not None:
                    raise ValueError("Multiple start (S) positions found in the grid.")
                start = (r, c)
            elif cell == GOAL_CELL:
                if goal is not None:
                    raise ValueError("Multiple goal (G) positions found in the grid.")
                goal = (r, c)
            elif cell not in (EMPTY, BLOCKED):
                raise ValueError(f"Unknown cell symbol '{cell}' at row {r}, col {c}.")

    # 7. Ensure Start and Goal were actually found
    if start is None:
        raise ValueError("No start position (S) found in the grid.")
    if goal is None:
        raise ValueError("No goal position (G) found in the grid.")

    # 8. Parse the beam width (k) from the final line
    k_str = raw_lines[rows + 1]
    if not k_str.isdigit() or int(k_str) <= 0:
        raise ValueError(f"k must be a positive integer, got '{k_str}'.")
    k = int(k_str)

    return grid, start, goal, k


# ---------------------------------------------------------------------------
# Grid printer — commented out before submission (for testing only)
# ---------------------------------------------------------------------------
# def display_grid(grid: list, path: list | None = None) -> None:
#     """
#     Print the grid to the console, optionally highlighting the solution path.
#     Cells on the path are shown as '*'.
#     Used during development/testing only.
#     """
#     path_set = set(path) if path else set()
#     rows = len(grid)
#     cols = len(grid[0])
#     header = "    " + "  ".join(str(c) for c in range(cols))
#     print(header)
#     print("   " + "---" * cols)
#     for r in range(rows):
#         row_str = f"{r} | "
#         for c in range(cols):
#             cell = grid[r][c]
#             if (r, c) in path_set and cell not in (START_CELL, GOAL_CELL):
#                 row_str += "*  "
#             else:
#                 row_str += f"{cell}  "
#         print(row_str)
#     print()


# ---------------------------------------------------------------------------
# Beam state printer — commented out before submission (for testing only)
# ---------------------------------------------------------------------------
# def print_beam_detail(beam: Beam, goal: tuple) -> None:
#     """
#     Print every entry in the beam with its path and heuristic.
#     Used during development/testing only.
#     """
#     if beam.is_empty():
#         print("[Beam] Empty — nothing to display.")
#         return
#     print(f"[Beam] {len(beam)}/{beam.capacity} states:")
#     for idx, (state, path) in enumerate(beam):
#         h = manhattan_distance(state, goal)
#         print(f"  [{idx}] state={state}  h={h}  path={path}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def main() -> None:
    """
    Entry point.
    1. Reads the warehouse grid from INPUT_FILE.
    2. Runs Local Beam Search.
    3. Prints results to the console and writes them to OUTPUT_FILE.
    """
    # Redirect stdout to Tee so console and file stay in sync
    tee = Tee(OUTPUT_FILE)
    sys.stdout = tee

    try:
        print("=" * 60)
        print("  Local Beam Search — Optimal Planner (G028)")
        print("=" * 60)
        print(f"  Input  file : {INPUT_FILE}")
        print(f"  Output file : {OUTPUT_FILE}")
        print("=" * 60)

        # ---- Read input ----
        grid, start, goal, k = read_input(INPUT_FILE)
        rows = len(grid)
        cols = len(grid[0])

        print(f"\nGrid       : {rows} x {cols}")
        print(f"Start      : {start}")
        print(f"Goal       : {goal}")
        print(f"Beam width : k = {k}")
        print(f"\nHeuristic  : Manhattan Distance  h(n) = |row_g - row_n| + |col_g - col_n|")
        print(f"Path cost  : 1 per move (Up / Down / Left / Right)\n")

        # ---- Display grid ----
        print("Grid layout:")
        header = "     " + "  ".join(str(c) for c in range(cols))
        print(header)
        print("    " + "---" * cols)
        for r in range(rows):
            print(f"  {r} | " + "  ".join(grid[r]))
        print()

        # ---- Run search ----
        path, cost = local_beam_search(grid, start, goal, k)

        # ---- Final summary ----
        print("\n" + "=" * 60)
        print("  SUMMARY")
        print("=" * 60)
        if path is not None:
            print(f"  Status     : SUCCESS")
            print(f"  Path       : {' -> '.join(str(p) for p in path)}")
            print(f"  Total cost : {cost} move(s)")
            print(f"  Path length: {len(path)} node(s)")
        else:
            print("  Status     : FAILURE — no path found.")
        print("=" * 60)

      # Add Complexity Analysis Printout Here
        print("-" * 60)
        print("  COMPLEXITY ANALYSIS")
        print("-" * 60)
        print("  Time Complexity:")
        print("    ├─ Successor generation : O(k * b) per iteration")
        print("    ├─ Candidate sorting    : O((k * b) log(k * b)) per iteration")
        print("    └─ Overall Time         : O(d * k * b * log(k * b))")
        print()
        print("  Space Complexity:")
        print("    └─ Peak memory          : O(k * b * d)")
        print()
        print("  where:")
        print("    k = beam width")
        print("    b = branching factor, maximum 4")
        print("    d = search depth / path length")
        print("=" * 60)
    except (FileNotFoundError, ValueError, TypeError, BeamFullError, BeamEmptyError, OSError, RuntimeError) as exc:
        print(f"\n[ERROR] {exc}")
    finally:
        tee.close()

if __name__ == "__main__":
    main()

  Local Beam Search — Optimal Planner (G028)
  Input  file : inputPS12.txt
  Output file : outputPS12.txt

Grid       : 5 x 5
Start      : (0, 0)
Goal       : (4, 4)
Beam width : k = 2

Heuristic  : Manhattan Distance  h(n) = |row_g - row_n| + |col_g - col_n|
Path cost  : 1 per move (Up / Down / Left / Right)

Grid layout:
     0  1  2  3  4
    ---------------
  0 | S  .  .  X  .
  1 | .  X  .  X  .
  2 | .  X  .  .  .
  3 | .  .  X  X  .
  4 | .  .  .  .  G

------------------------------------------------------------
Initialisation
  Start : (0, 0)   h = 8
  Beam  : [(0, 0)]
------------------------------------------------------------

Iteration 1
  Current beam states : [(0, 0)]
  Heuristic values    : [8]

  Successors generated (unique) : [(1, 0), (0, 1)]
  After sorting by heuristic    : [(1, 0), (0, 1)]
  Heuristic values              : [7, 7]

  Selected k=2 beam state(s) : [(1, 0), (0, 1)]
  Heuristic values             : [7, 7]

Iteration 2
  Current beam states : [(1, 0), (